In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('../data/adn_boca_real_features.csv')

print(f"Dataset: {df.shape}")
print(df.isnull().sum())

In [ ]:
le = LabelEncoder()
df['posicion_encoded'] = le.fit_transform(df['posicion'])

features = ['posicion_encoded', 'edad', 'partidos',
            'goles', 'asistencias', 'pases_precisos',
            'goles_por_partido', 'participacion_gol', 'experiencia']

X = df[features]
y = df['etiqueta']

print(f"Features: {features}")
print(f"Etiquetas:\n{y.value_counts()}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import joblib
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: escalar + logistic con L1 (Lasso)
# L1 hace seleccion automatica de features → menos overfitting
modelo = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        l1_ratio=1,
        solver='saga',
        C=0.1,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=['No encaja', 'Perfil Boca']))

cv_scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")

coefs = modelo.named_steps['lr'].coef_[0]
print("\nCoeficientes L1 (seleccion automatica de features):")
for f, c in sorted(zip(features, coefs), key=lambda x: -abs(x[1])):
    print(f"  {f:25s} {c:+.4f}")

In [ ]:
# Guardar modelo
joblib.dump(modelo, '../models/modelo_logistic_l1.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
print("Modelo guardado en models/modelo_logistic_l1.pkl")

In [ ]:
# Prediccion sobre todo el dataset
df_pred = df.copy()
df_pred['probabilidad'] = modelo.predict_proba(X)[:, 1]
df_pred['prediccion'] = modelo.predict(X)

resultado = df_pred[df_pred['prediccion'] == 1]\
    .sort_values('probabilidad', ascending=False)

print("Top 10 jugadores con perfil Boca (L1):")
print(resultado[['nombre', 'posicion', 'temporada', 'goles',
                 'asistencias', 'probabilidad']].head(10).to_string(index=False))